<a href="https://colab.research.google.com/github/jbarrasa/goingmeta/blob/main/session43/GM_S3_5_AgentMemory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install neo4j-agent-memory[all] gliner

In [ ]:
import asyncio
from google.colab import userdata
from pydantic import SecretStr
from openai import AsyncOpenAI
from neo4j_agent_memory import MemoryClient, MemorySettings, Neo4jConfig, ToolCallStatus
import os

os.environ.setdefault("OPENAI_API_KEY", userdata.get('OPENAI_API_KEY'))
 #("OPENAI_API_KEY")

# Configure settings
settings = MemorySettings(
    neo4j=Neo4jConfig(
        uri=userdata.get('NEO4J_URL'),
        username=userdata.get('NEO4J_USR'),
        password=SecretStr(userdata.get('NEO4J_PWD')),
    )
)


## Short term memory

In [ ]:
async with MemoryClient(settings) as memory:

  # Add messages to a conversation
  await memory.short_term.add_message(
      session_id="user-123",
      role="user",
      content="I'm looking for a restaurant"
  )

  await memory.short_term.add_message(
      session_id="user-123",
      role="assistant",
      content="That's great, what's your favourite cuisine?"
  )

  await memory.short_term.add_message(
      session_id="user-123",
      role="user",
      content="I love Mediterranean cuisine"
  )

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
 async with MemoryClient(settings) as memory:
  # Get conversation history
  conversation = await memory.short_term.get_conversation("user-123")
  for msg in conversation.messages:
      print(f"{msg.role}: {msg.content}")

  # Search past messages
  results = await memory.short_term.search_messages("restaurants")
  for msg in results:
      print(f"{msg.role}: {msg.content}")


MessageRole.USER: I'm looking for a restaurant
MessageRole.ASSISTANT: That's great, what's your favourite cuisine?
MessageRole.USER: I love Mediterranean cuisine
MessageRole.USER: I'm looking for a restaurant


## Long term memory

In [ ]:
async with MemoryClient(settings) as memory:
  # Add entities with POLE+O types and subtypes
  entity = await memory.long_term.add_entity(
      name="John Smith",
      entity_type="PERSON",  # POLE+O type
      subtype="INDIVIDUAL",  # Optional subtype
      description="A customer who loves Italian food"
  )

  # Add preferences
  pref = await memory.long_term.add_preference(
      category="food",
      preference="Prefers vegetarian options",
      context="When dining out"
  )

  # Add facts with temporal validity
  from datetime import datetime
  fact = await memory.long_term.add_fact(
      subject="John",
      predicate="works_at",
      obj="Acme Corp",
      valid_from=datetime(2023, 1, 1)
  )

In [ ]:
async with MemoryClient(settings) as memory:
  # Search for relevant entities
  entities = await memory.long_term.search_entities("Smith")
  for entity in entities:
    print(entity.name, entity.type, entity.description)

John Smith PERSON A customer who loves Italian food


## Reasoning Memory

In [ ]:
async with MemoryClient(settings) as memory:
  # Start a reasoning trace (optionally linked to a triggering message)
  trace = await memory.reasoning.start_trace(
      session_id="user-123",
      task="Find a restaurant recommendation",
      triggered_by_message_id= "fc1418d1-a7db-4ff6-964e-057ea7734edd",  # Optional: link to message
  )

  # Add reasoning steps
  step = await memory.reasoning.add_step(
      trace.id,
      thought="I should search for nearby restaurants",
      action="search_restaurants"
  )

  # Record tool calls (optionally linked to a message)
  await memory.reasoning.record_tool_call(
      step.id,
      tool_name="search_api",
      arguments={"query": "Italian restaurants"},
      result=["La Trattoria", "Pasta Palace"],
      status=ToolCallStatus.SUCCESS,
      duration_ms=150,
      message_id="fc1418d1-a7db-4ff6-964e-057ea7734edd",  # Optional: link tool call to message
  )

  # Complete the trace
  await memory.reasoning.complete_trace(
      trace.id,
      outcome="Recommended La Trattoria",
      success=True
  )

  # Link an existing trace to a message (post-hoc)
  #await memory.reasoning.link_trace_to_message(trace.id, message.id)

In [ ]:
async with MemoryClient(settings) as memory:
  # Find similar past tasks
  similar = await memory.reasoning.get_similar_traces("recommending restaurants")
  for t in similar:
    print(t.task, t.created_at, t.completed_at)


Find a restaurant recommendation 2026-02-03 16:35:50.680485 2026-02-03 16:34:01.859000+00:00


## Conversation Summaries

In [ ]:
async with MemoryClient(settings) as memory:
  # Basic summary (no LLM required)
  summary = await memory.short_term.get_conversation_summary("user-123")
  print(summary.summary)
  print(f"Messages: {summary.message_count}")
  print(f"Key entities: {summary.key_entities}")

  openai_client = AsyncOpenAI(
      api_key=userdata.get("OPENAI_API_KEY"),
  )
  # With custom LLM summarizer
  async def my_summarizer(transcript: str) -> str:
      # Your LLM call here
      response = await openai_client.chat.completions.create(
          model="gpt-4o-mini",
          messages=[
              {"role": "system", "content": "Summarize this conversation concisely."},
              {"role": "user", "content": transcript}
          ]
      )
      return response.choices[0].message.content

  summary = await memory.short_term.get_conversation_summary(
      "user-123",
      summarizer=my_summarizer,
      include_entities=True,
  )

Conversation with 3 messages (2 user, 1 assistant). Started with: "I'm looking for a restaurant"
Messages: 3
Key entities: ['Mediterranean']


In [ ]:
print(summary)

session_id='user-123' summary='The user is looking for a restaurant and expresses a preference for Mediterranean cuisine.' message_count=3 time_range=(datetime.datetime(2026, 2, 3, 16, 13, 25, 681000, tzinfo=<UTC>), datetime.datetime(2026, 2, 3, 16, 13, 43, 811000, tzinfo=<UTC>)) key_entities=['Mediterranean'] key_topics=['looking', 'restaurant', 'expresses', 'preference', 'mediterranean'] generated_at=datetime.datetime(2026, 2, 3, 16, 41, 33, 789475)


Other Experiments (not all covered in the Going Meta session)

In [ ]:
import asyncio
from google.colab import userdata
from pydantic import SecretStr
from neo4j_agent_memory import MemoryClient, MemorySettings, Neo4jConfig
import os

os.environ.setdefault("OPENAI_API_KEY", userdata.get('OPENAI_API_KEY'))
 #("OPENAI_API_KEY")

# Configure settings
settings = MemorySettings(
    neo4j=Neo4jConfig(
        uri=userdata.get('NEO4J_URL'),
        username=userdata.get('NEO4J_USR'),
        password=SecretStr(userdata.get('NEO4J_PWD')),
    )
)

In [ ]:
async with MemoryClient(settings) as memory:
  await memory.long_term.add_preference(
      category="interests",
        preference="Loves football",
        context="When watching sports")

In [ ]:
async with MemoryClient(settings) as memory:
  await memory.short_term.add_message(
    session_id="user-123",
    role="user",
    content="I'm looking for football game to watch"
)

In [ ]:
from datetime import datetime
async with MemoryClient(settings) as memory:
  fact = await memory.long_term.add_fact(
    subject="JB",
    predicate="works_at",
    obj="Neo4j",
    valid_from=datetime(2015, 1, 1)
)

In [ ]:
from datetime import datetime
async with MemoryClient(settings) as memory:
  fact = await memory.long_term.add_fact(
    subject="JB",
    predicate="works_at",
    obj="Neo4j",
    valid_until=datetime(2027, 1, 1)
)

In [ ]:
async with MemoryClient(settings) as memory:
        # Store a conversation message
        await memory.short_term.add_message(
            session_id="user-123",
            role="user",
            content="Hi, I'm JB and I support Real Madrid"
        )

In [ ]:
# Use the memory client
async with MemoryClient(settings) as memory:
    # Store a conversation message
    await memory.short_term.add_message(
        session_id="user-123",
        role="user",
        content="Hi, I'm John and I love Italian food!"
    )

    # Add a preference
    await memory.long_term.add_preference(
        category="food",
        preference="Loves Italian cuisine",
        context="Dining preferences"
    )

    # Search for relevant memories
    preferences = await memory.long_term.search_preferences("restaurant recommendation")
    for pref in preferences:
        print(f"[{pref.category}] {pref.preference}")

    # Get combined context for an LLM prompt
    context = await memory.get_context(
        "What restaurant should I recommend?",
        session_id="user-123"
    )
    print(context)

In [ ]:
async with MemoryClient(settings) as memory:
  entities = await memory.long_term.search_preferences("sports")
  for entity in entities:
    print(entity.category, entity.preference, entity.context, entity.linked_entities)

In [ ]:
import asyncio
from google.colab import userdata
from pydantic import SecretStr
from neo4j_agent_memory import MemoryClient, MemorySettings, Neo4jConfig
import os

os.environ.setdefault("OPENAI_API_KEY", userdata.get('OPENAI_API_KEY'))
 #("OPENAI_API_KEY")

# Configure settings
settings = MemorySettings(
    neo4j=Neo4jConfig(
        uri=userdata.get('NEO4J_URL'),
        username=userdata.get('NEO4J_USR'),
        password=SecretStr(userdata.get('NEO4J_PWD')),
    )
)

# Use the memory client
async with MemoryClient(settings) as memory:
    # Store a conversation message
    # await memory.short_term.add_message(
    #     session_id="user-124",
    #     role="user",
    #     content="Hi, I'm JB and I support Real Madrid!"
    # )

    # Add a preference
    await memory.long_term.add_preference(
        category="sports",
        preference="Loves football",
        context="Football allegiance"
    )

    # # Search for relevant memories
    # preferences = await memory.long_term.search_preferences("football game recommendation")
    # for pref in preferences:
    #     print(f"[{pref.category}] {pref.preference}")

    # # Get combined context for an LLM prompt
    # context = await memory.get_context(
    #     "What game should I watch?",
    #     session_id="user-124"
    # )
    print(context)

In [ ]:
from neo4j_agent_memory.extraction import create_extractor
from neo4j_agent_memory.config import ExtractionConfig

# Create the default pipeline (spaCy → GLiNER → LLM)
config = ExtractionConfig(
    extractor_type="pipeline",
    enable_spacy=True,
    enable_gliner=True,
    enable_llm_fallback=True,
    merge_strategy="confidence",  # Keep highest confidence per entity
)

extractor = create_extractor(config)
result = await extractor.extract("John Smith works at Acme Corp in New York.")

In [ ]:
from neo4j_agent_memory  import ToolCallStatus
from neo4j_agent_memory.config import ExtractionConfig

user_message_id = "9ba85c5c-3176-4125-b4a5-3850ec297eeb"

async with MemoryClient(settings) as memory:
  # Start a reasoning trace (optionally linked to a triggering message)
  trace = await memory.reasoning.start_trace(
      session_id="user-123",
      task="Find a restaurant recommendation",
      triggered_by_message_id=user_message_id,  # Optional: link to message
  )

  # Add reasoning steps
  step = await memory.reasoning.add_step(
      trace.id,
      thought="I should search for nearby restaurants",
      action="search_restaurants"
  )

  # Record tool calls (optionally linked to a message)
  await memory.reasoning.record_tool_call(
      step.id,
      tool_name="search_api",
      arguments={"query": "Italian restaurants"},
      result=["La Trattoria", "Pasta Palace"],
      status=ToolCallStatus.SUCCESS,
      duration_ms=150,
      message_id=user_message_id,  # Optional: link tool call to message
  )

  # Complete the trace
  await memory.reasoning.complete_trace(
      trace.id,
      outcome="Recommended La Trattoria",
      success=True
  )

  # Find similar past tasks
  similar = await memory.reasoning.get_similar_traces("restaurant recommendation")

  # Link an existing trace to a message (post-hoc)
  #await memory.reasoning.link_trace_to_message(trace.id, message.id)

In [ ]:
transcript = """
Going Meta: Two Years of Knowledge Graphs

We take a look back at the past episodes of Going Meta, specifically those where we cover knowledge graphs. This is our recap of Season 2 of Going Meta, a good year after I wrote the last episode of Season 1.

But first, I want to thank you for your continued support over the years (crazy that I can say that now). Our GitHub repository, where we gather all the assets (code, queries, datasets, ontologies, notebooks, etc.) for each episode, is a great source for you.


With Season 2 coming to an end, we wanted to pick out one integral pillar of Going Meta: knowledge graphs. Plenty of episodes cover them across the season (and beyond in Season 1), so we wanted to offer a guide on what topics we covered in each episode in case you’re new to the series or you missed an episode.

Foundations
Toward the end of Season 1, we explored how semantics (another theme of Going Meta) can be captured in two fundamentally different ways: explicitly through knowledge graphs and implicitly through vector embeddings. We demonstrated how the explicit graph-based semantics offer explanability and rich exploration capabilities, while vector embeddings provide robust semantic search. These approaches are highly complementary rather than competing: Graphs provide interpretable structure and context, while vectors enable efficient semantic similarity matching. This approach creates powerful retrieval-augmented generation (RAG) systems that move beyond pure similarity to actual relevance.

Episodes:
S01 Ep21 — Explicit and Implicit Semantics
S01 Ep22— Basic RAG and RAG with Knowledge Graphs

Retrieval Patterns
We explored sophisticated retrieval techniques that combine vector search with graph traversals. Starting with basic vector search that lands on relevant nodes, the approach then navigates through graph relationships to contextualize and enrich results.

We introduced ontology-driven dynamic exploration, where the type of entity discovered determines the navigation strategy. Later episodes compared multiple retrieval methods available in Neo4j (vector, full-text, geospatial, and Cypher generation) and evolved toward agentic approaches where LLMs autonomously select appropriate retrieval tools based on query requirements. The progression moved from one-shot retrieval to multi-tool orchestration, demonstrating how to offer retrieval capabilities as functions that agents can intelligently combine.

Episodes:
S01 Ep22 — RAG with Vectors and Traversals
S01 Ep24 — Adding Ontologies to the Mix
S02 Ep06 — Comparing Retrieval Methods
S02 Ep07 — Function Calling (Tools)

LLMs as Assistants for Domain Modeling
These episodes pioneered the use of LLMs for graph domain modeling, exploring how AI could assist in the traditionally complex task of designing graph schemas from structured data. We showed the LLMs’ ability to identify entities, relationships, and logical data organization patterns — essentially performing entity-relationship modeling.

We then implemented an agentic workflow in which one LLM acts as a modeling expert while another serves as a critic, providing feedback and iterating to improve model quality. This approach proved that LLMs could produce graph models indistinguishable from expert-designed ones. This is now widely known as LLM-as-a-Critic and Agentic AI.

Episodes:
S01 Ep25 —Learn a Graph Mode from a Denormalized Dataset
S01 Ep27 — Call a Critic, Get Feedback, and Iterate!

Knowledge Graph Construction
We spent most of the time on this topic and covered Neo4j’s unique dual-graph approach to knowledge graph construction, combining document/lexical graphs (representing document structure) with domain graphs (representing extracted entities and relationships). Our key message here is the importance of using ontologies or target schemas as guardrails. Knowledge graphs without well-defined schemas can easily lead to unmanageable results.

Across different episodes, we covered no-code visual approaches and programmatic methods for extracting knowledge from unstructured data (PDFs, web pages) and structured data (CSV files, databases). A key insight was the “mixed data” approach, which showed how a single ontology can drive construction from structured and unstructured sources simultaneously, with structured data providing scaffolding for unstructured content integration.

Episodes:
S02 Ep01 — From Unstructured Data (No Code)
S02 Ep02 and Ep03 — From Unstructured Data (Programmatic)
S02 Ep05 — From Structured Data (No Code)
S01 Ep05 and S02 Ep07 — From Structured Data (Programmatic)
"""

In [ ]:
from neo4j_agent_memory.extraction import create_extractor
from neo4j_agent_memory.config import ExtractionConfig

# Create the default pipeline (spaCy → GLiNER → LLM)
config = ExtractionConfig(
    extractor_type="pipeline",
    enable_spacy=True,
    enable_gliner=True,
    enable_llm_fallback=True,
    merge_strategy="confidence",  # Keep highest confidence per entity
)

extractor = create_extractor(config)
result = await extractor.extract(transcript)

In [ ]:
from neo4j_agent_memory.extraction import ExtractorBuilder

# Use the fluent builder API
extractor = (
    ExtractorBuilder()
    .with_spacy(model="en_core_web_sm")
    .with_gliner(model="urchade/gliner_medium-v2.1", threshold=0.5)
    .with_llm_fallback(model="gpt-4o-mini")
    .merge_by_confidence(threshold= 0.8)
    .build()
)

result = await extractor.extract(transcript)
for entity in result.entities:
    print(f"{entity.name}: {entity.type} (confidence: {entity.confidence:.2f})")

In [ ]:
for entity in result.filter_invalid_entities().entities:
    print(f"{entity.name}: {entity.type} ({entity.confidence:.0%})")

In [ ]:
from neo4j_agent_memory.extraction import (
    GLiNEREntityExtractor,
    get_schema,
    list_schemas,
)

# List available pre-defined schemas
print(list_schemas())
# ['poleo', 'podcast', 'news', 'scientific', 'business', 'entertainment', 'medical', 'legal']

# Create extractor with domain schema
extractor = GLiNEREntityExtractor.for_schema("poleo", threshold=0.45)

# Or use with the ExtractorBuilder
from neo4j_agent_memory.extraction import ExtractorBuilder

extractor = (
    ExtractorBuilder()
    .with_spacy()
    .with_gliner_schema("poleo", threshold=0.5)
    .with_llm_fallback()
    .merge_by_confidence()
    .build()
)

# Extract entities from domain-specific content
result = await extractor.extract(transcript)
for entity in result.filter_invalid_entities().entities:
    print(f"{entity.name}: {entity.type} {entity.subtype} ({entity.confidence:.0%})")